# Day 015 Solution — Multi-Turn CLI Chatbot

In [ ]:
import ollama

MODEL = "llama3.2"

In [ ]:
# ── Core history functions ───────────────────────────────────────────────────

def append_turn(
    history: list[dict],
    user_text: str,
    assistant_text: str,
) -> list[dict]:
    """Return a new history list with one user+assistant turn appended."""
    return history + [
        {"role": "user",      "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]


def chat_turn(
    history: list[dict],
    user_input: str,
    model: str = MODEL,
) -> tuple[str, list[dict]]:
    """Execute one conversational turn and return (reply, updated_history)."""
    messages = history + [{"role": "user", "content": user_input}]
    response = ollama.chat(model=model, messages=messages)
    reply = response["message"]["content"]
    return reply, append_turn(history, user_input, reply)


def truncate_history(
    history: list[dict],
    max_turns: int = 10,
) -> list[dict]:
    """Keep the system prompt and the most recent max_turns user/assistant pairs."""
    if not history:
        return []
    if history[0]["role"] == "system":
        system = [history[0]]
        tail   = history[1:]
    else:
        system = []
        tail   = history
    return system + tail[-(max_turns * 2):]


def reset_history(history: list[dict]) -> list[dict]:
    """Return a new history containing only the system message (if present)."""
    if history and history[0]["role"] == "system":
        return [history[0]]
    return []


def format_history(history: list[dict]) -> str:
    """Render conversation history as a human-readable transcript."""
    labels = {"user": "You", "assistant": "Bot"}
    lines = []
    for msg in history:
        if msg["role"] == "system":
            continue
        label = labels.get(msg["role"], msg["role"].capitalize())
        lines.append(f"{label}: {msg['content']}")
    return "\n".join(lines)

In [ ]:
SYSTEM_PROMPT = (
    "You are a concise, friendly assistant. "
    "Keep all answers to one or two sentences."
)

# Scripted demo — shows all 5 functions without requiring stdin
print("=== Scripted Demo: 3-Turn Conversation ===\n")

history = [{"role": "system", "content": SYSTEM_PROMPT}]

questions = [
    "What is the capital of France?",
    "And what is it famous for?",
    "What were my two questions?",
]

for q in questions:
    reply, history = chat_turn(history, q, model=MODEL)
    history = truncate_history(history, max_turns=10)
    print(f"You: {q}")
    print(f"Bot: {reply}\n")

print("=== Conversation Transcript ===")
print(format_history(history))

In [ ]:
print("\n=== Demo: /reset command ===")
print(f"Before reset: {len(history)} messages")
history = reset_history(history)
print(f"After reset:  {len(history)} message (system prompt only)")
print(f"System: {history[0]['content']}")

In [ ]:
def run_chatbot(
    system_prompt: str = SYSTEM_PROMPT,
    model: str = MODEL,
    max_turns: int = 10,
) -> None:
    """
    Run a multi-turn CLI chatbot until the user types /quit.
    
    This function uses input() and is intended for interactive use.
    Run it in a terminal, not in the gate checker.
    """
    history = [{"role": "system", "content": system_prompt}]
    print("Chatbot ready. Commands: /quit  /reset  /history")
    print("-" * 50)

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue

        if user_input.startswith("/"):
            if user_input == "/quit":
                print("Goodbye!")
                break
            elif user_input == "/reset":
                history = reset_history(history)
                print("Bot: Conversation reset.")
            elif user_input == "/history":
                transcript = format_history(history)
                print(transcript if transcript else "(no history yet)")
            else:
                print(f"Unknown command: {user_input}")
            continue

        reply, history = chat_turn(history, user_input, model)
        history = truncate_history(history, max_turns)
        print(f"Bot: {reply}")

# To run interactively in a terminal:
# run_chatbot()
print("run_chatbot() defined — call it in a terminal for interactive use.")

In [ ]:
print("Day 015 project complete — multi-turn chatbot with history management.")